# UCREL-NLP Summer School 2026

![header](https://r2.epsiotapi.com/Summer-School-2026/header_img.png)

# Introduction to LLM Prompting Techniques

## Introduction

In this tutorial, we will learn how to:
1. **Obtain an API Key from [openrouter.ai](https://openrouter.ai) and configure it in Google Colab**
2. **Interact with Large Language Models (LLMs) through the OpenAI API**
3. **Apply different prompting techniques when working with LLMs**
4. **Understand the basic concepts of Chain-of-Thought (CoT) prompting and Retrieval-Augmented Generation (RAG)**
5. **Practise the techniques you have learned through hands-on tasks and experiments**


To achieve the objectives above, we will follow the steps below:

- [**Section 1: Preparation**](#preparation) – Download the materials, apply the API key from [openrouter.ai](https://openrouter.ai), and configure the environment variables in Google Colab.

- [**Section 2: Prompting LLM with RTCF Framework**](#rtcf) – Provide LLMs with detailed and relevant information using a structured prompt.

- [**Section 3: Few-Shot Prompting**](#few-shot) – Improve LLM performance by providing example inputs and outputs.

- [**Section 4: Constraints**](#constraints) – Impose constraints on the behaviour of LLMs.

- [**Section 5: Chain-of-Thought (CoT) Prompting**](#cot) – Encourage LLMs to make their reasoning process explicit.

- [**Section 6: Retrieval-Augmented Generation (RAG)**](#rag) - Adding specific data to the LLMs.


<a name="preparation"></a>
## Section 1: Preparation

In this section, you will learn how to:

1. Create an account on OpenRouter and obtain an API key.
2. Configure your API key and interact with LLMs through the API.

### Install the OpenAI SDK and import the required Python libraries

In [ ]:
!pip install openai

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

### Configure your API key in Colab

1. Open the **🔑 Secrets** panel on the left side of Colab, add a new secret named `OPENROUTER_KEY`, and set its value to your OpenRouter API key.
2. Grant this notebook access to the secret.
3. Run the code below. If the configuration is successful, you should be able to retrieve and view the list of models available through OpenRouter.

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_KEY')
)

# list the model provided by OpenRouter
models = client.models.list()
for model in models.data:
    print(model.id)

Please choose a model from the list above to use in the following experiments.

In [ ]:
model_alias = "google/gemma-4-31b-it:free"

The OpenAI API is designed as a stateless, which means that the server does not store conversation history.
As a result, the client is responsible for maintaining the conversation history itself.

When sending a request to the API, the `messages` field should contain not only the current user query,
but also any previous user messages and assistant responses that are necessary for maintaining conversational context.

The OpenAI API supports three message roles:

1. `system` – System-level instructions that define the assistant's behaviour, personality, or response style.
2. `user` – Messages sent by the user to the LLM.
3. `assistant` – Responses generated by the LLM.

In [ ]:
def send_message(prompt: str, sys_prompt: str="You are a helpful AI assistant.",
                 model_alias=model_alias, temperature=0, max_tokens=1600, seed=20260701):
  response = client.chat.completions.create(
    model=model_alias,
    messages=[
        {
            "role": "system",
            "content": sys_prompt
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=temperature,  # controls randomness of output (0 = deterministic/focused, higher values like 1-2 = more random/creative)
    max_tokens=max_tokens,    # maximum number of tokens the model can generate in the response
    seed=seed                 # fixes the random number generator for (mostly) reproducible outputs across calls with the same parameters
  )
  print(response.choices[0].message.content)

We can now ask the LLM to perform a simple task, such as designing a fitness meal plan for the user.


In [ ]:
system_instruction = "You are a helpful assistant."

send_message("Please design a fitness meal plan for me.", system_instruction)

<a name="rtcf"></a>
## Section 2: Prompting LLM with RTCF Framework

When interacting with LLMs, the way a task is described in natural language has a direct impact on the model's performance.
Communicating your requirements clearly helps the model better understand your intent and generate useful responses.
The goal of prompt engineering is to construct clear, precise, and effective prompts that enable LLMs to perform tasks more reliably.

We recommend using the RTCF framework to organise prompts. Under this framework, a prompt consists of four parts:

1. **R (Role)** – Who would you like the model to act as?
2. **T (Task)** – What task would you like the model to perform?
3. **C (Context)** – What background information, constraints, or details can help the model complete the task more accurately?
4. **F (Format)** – In what format should the model present its output?

In [ ]:
prompt_role = "You are a professional sports nutritionist with expertise in fitness fat-loss and muscle-building meal planning.\n"

prompt_task = "Please create a personalized breakfast menu based on the user’s physical condition and fitness goals, complete with preparation steps and nutritional information. \n"

prompt_context = """
User Information: Female, height 165 cm, weight 58 kg
Fitness Goals: Fat loss
Dietary Preference: Doesn't eat egg \n
"""

prompt_format = """Output the result in the following format:
1. Meal Plan:
2. Ingredients:
  Step 1:
  Step 2:
  ...
3. Preparation Steps:
4. Nutrition Information: calories and protein content.
"""

# Combine the instructions.
system_instruction = prompt_role + prompt_task + prompt_context + prompt_format
print(system_instruction)

In [ ]:
send_message("Please design a fitness meal plan for me.", system_instruction)

<a name="few-shot"></a>
## Section 3: Few-Shot Prompting

When it is difficult to describe our requirements clearly in natural language, we can instead provide the LLM with a few worked examples.
The model can learn the task, output style, and response format directly from the provided demonstrations.

[Here](https://r2.epsiotapi.com/Summer-School-2026/meal_plan_example.md) is a sample fitness meal plan.

We will include this example in the prompt to guide the model when it generates a meal plan for the user.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # mount the google drive

In [ ]:
with open("/content/drive/MyDrive/Summer School 2026/materials/meal_plan_example.md", "r") as f:
    example = f.read()
print(example)

You also can upload the file from the loacl by using the following code

In [ ]:
# You also can upload the file from local.
from google.colab import files

_ = files.upload()
file_name = "meal_plan_example.md"

with open(file_name, "r") as f:
    example = f.read()
print(example)

In [ ]:
prompt_few_shot = (
    "Below is a sample fitness breakfast menu. Please use follow the format to create a customized fitness breakfast menu for the user. \n"
    + example + "\n"
)

system_instruction = prompt_role + "\n" + prompt_few_shot
print(system_instruction)

In [ ]:
text = "Please design a fitness meal plan for me." + prompt_context
print(text)

In [ ]:
send_message(text, system_instruction)

<a name="constraints"></a>
## Section 4: Constraints

LLMs are naturally inclined to be helpful and accommodating, which often leads them to generate excessive or unnecessary information.

To make their outputs better align with users' actual needs, it is not enough simply to tell them what to do—we must also learn how to tell them what **not** to do.

In other words, effective prompting is not only about specifying the desired behavior, but also about defining clear boundaries against undesirable behaviour.
In some cases, it may even be necessary to restrict the range of acceptable outputs explicitly.
By imposing such constraints, we can reduce unwanted variability and steer the model towards responses that better match our expectations

In this section, we will explore three common types of constraints:

1. **Content constraints** – restrict what information the model should or should not include.
2. **Numerical constraints** – constrain numerical outputs, ranges, quantities, or calculation requirements.
3. **Format constraints** – specify how the output should be structured and presented.

> Have you ever used an AI agent? Since AI agents are often required to perform relatively complex tasks,
> they may inevitably exhibit behaviours that the user did not anticipate.
> Therefore, placing appropriate constraints on an AI agent is extremely important.

In [ ]:
system_instruction = """
You are a professional sports nutritionist.
Please create a customized fitness breakfast menu for the user.
Keep it as concise as possible.
"""

text = "My primary fitness goal is to build muscle. Please design a fitness meal plan for me."
send_message(text, system_instruction)

#### Content constraints

In [ ]:
content_constraints = """
Please do not provide the user with multiple options.
Do not use protein powder in the meal plan.
"""

constrained_instruction = system_instruction + content_constraints
print(constrained_instruction)

In [ ]:
send_message(text, constrained_instruction)

#### Numerical constraints

In [ ]:
numerical_constraints = """
The meal should provide more than 20 g of protein.
The total ingredient cost should not exceed £5.
"""

constrained_instruction = system_instruction + numerical_constraints
print(constrained_instruction)

In [ ]:
send_message(text, constrained_instruction)

#### Format constraint

In [ ]:
format_constraints = """
The entire menu should not exceed 100 words in total.
Output the result in plain text only. Do not use any Markdown formatting.
"""

constrained_instruction = system_instruction + format_constraints

print(constrained_instruction)

In [ ]:
send_message(text, constrained_instruction)

### Appendix

[Json Schema](https://json-schema.org/)